In [ ]:
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

resources_path = 'resources/'
output_path = 'output/'

# Importing the dataset

In [ ]:
# import the aligned dataset
aligned  = pd.read_pickle(resources_path + 'en80jours_aligned.pkl')
print(aligned.columns)
aligned.sample(3)

In [ ]:
# terms with freq > 5
frequent_fr_norm = [term for term, freq in aligned.fr_norm.value_counts().items() if freq > 5]
frequent_en_norm = [term for term, freq in aligned.en_norm.value_counts().items() if freq > 5]
frequent_de_norm = [term for term, freq in aligned.de_norm.value_counts().items() if freq > 5]
frequent_sb_norm = [term for term, freq in aligned.sr_norm.value_counts().items() if freq > 5]

# keep only rows where all 4 terms are in the frequent lists
print(f"Original size: {len(aligned)}")
aligned_frequent = aligned[
    aligned.fr_norm.isin(frequent_fr_norm) &
    aligned.en_norm.isin(frequent_en_norm) &
    aligned.de_norm.isin(frequent_de_norm) &
    aligned.sr_norm.isin(frequent_sb_norm)
]
print(f"Size after filtering: {len(aligned_frequent)}")

# keep only rows where all 4 terms are not None or empty
aligned_frequent = aligned_frequent[
    aligned_frequent.fr_norm.notna() & (aligned_frequent.fr_norm != '') & (aligned_frequent.fr_norm != 'None') &
    aligned_frequent.en_norm.notna() & (aligned_frequent.en_norm != '') & (aligned_frequent.en_norm != 'None') &
    aligned_frequent.de_norm.notna() & (aligned_frequent.de_norm != '') & (aligned_frequent.de_norm != 'None') &
    aligned_frequent.sr_norm.notna() & (aligned_frequent.sr_norm != '') & (aligned_frequent.sr_norm != 'None')
]
print(f"Size after removing empty/None: {len(aligned_frequent)}")

# 1) Modelling the psychological space of spatial relations

## Importing pile sorting results

$N=35$ participants, $K=30$ cards to sort

In [ ]:
N = 35
K = 30

In [ ]:
all_exp = [ 
    pd.read_csv(resources_path + 'pile_sorting_results/' + f'subject_{i}_piles.csv')
    for i in range(N)
]
nb_piles  = [piles.shape[1] for piles in all_exp]

# distribution of number of piles
plt.figure(figsize=(10, 2))
plt.hist(nb_piles, bins=range(1, max(nb_piles)+1), align='left', rwidth=0.8)
plt.xlabel('Number of Piles')
plt.xticks(range(1, max(nb_piles)+1))
plt.ylabel('Count')
plt.show()

In [ ]:
def piles_to_matrix(exp: pd.DataFrame) -> np.ndarray:
    """Convert pile-sorting dataframe to a count matrix"""
    n_items = K
    count_matrix = np.zeros((n_items, n_items))

    for col in exp.columns:
        for i in exp[col].dropna():
            for j in exp[col].dropna():
                count_matrix[int(i), int(j)] = 1
                count_matrix[int(j), int(i)] = 1

    assert np.allclose(count_matrix, count_matrix.T), "Count matrix is not symmetric"

    return count_matrix

count_matrices = [piles_to_matrix(exp) for exp in all_exp]
aggregated_matrix = np.mean(count_matrices, axis=0)
np.fill_diagonal(aggregated_matrix, 1)

# visualize the aggregated similarity matrix as a heatmap
plt.figure()
sns.heatmap(aggregated_matrix, cmap='viridis', cbar=True, linewidth=.5)
plt.show()


## Training a model to predict the similarities $\widehat{sim}_{i,j}$

- human similarities $sim_{i,j}$ : `human_sim`
- contextual embeddings $F_{i,j}$ : `F_pairs`

In [ ]:
from scipy.stats import spearmanr
from scipy.special import softmax
from sklearn.metrics import make_scorer
from sklearn.metrics.pairwise import cosine_similarity as cosim
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline

In [ ]:
# human pairwise similarities
human_sim = []
for i in range(30):
    for j in range(i+1, 30):
        human_sim.append(aggregated_matrix[i, j])
human_sim = np.array(human_sim)

In [ ]:
with open(resources_path + 'centroids_embeddings_list.pkl', 'rb') as f:
    centroids_embeddings_list = pickle.load(f)

# pairs of input embeddings
F_pairs = []
for i in range(len(centroids_embeddings_list)):
    for j in range(i+1, len(centroids_embeddings_list)):
        F_pairs.append((centroids_embeddings_list[i], centroids_embeddings_list[j]))
F_pairs = np.array(F_pairs)

### baseline : cosine similarity

In [ ]:
sim_hat = [
    cosim(F_pairs[i][0].reshape(1, -1), F_pairs[i][1].reshape(1, -1))[0][0]
    for i in range(len(F_pairs))
]

corr, p_value = spearmanr(human_sim, sim_hat)
print(f"Spearman correlation: {corr:.4f}, p-value: {p_value:.4e}")

### Ridge regression

In [ ]:
# Custom Transformer for Interaction Features
class EmbeddingInteractions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X shape: (n_samples, 2, embedding_dim)
        u, v = X[:, 0, :], X[:, 1, :]
        
        # Features: Hadamard (alignment) and L1 (distance)
        prod = u * v
        diff = np.abs(u - v)
        
        # Cosine similarity
        norm_u = np.linalg.norm(u, axis=1, keepdims=True)
        norm_v = np.linalg.norm(v, axis=1, keepdims=True)
        # Add small epsilon to avoid division by zero
        cosine = np.sum(u * v, axis=1, keepdims=True) / (norm_u * norm_v + 1e-10)
        
        return np.hstack([prod, diff, cosine])

def spearman_metric(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

# Pipeline Definition
pipe = Pipeline([
    ('features', EmbeddingInteractions()),
    ('scaler', StandardScaler()),
    ('model', Ridge(solver='svd')) 
])

# Nested Cross-Validation Setup
# Inner CV: For hyperparameter optimization
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Outer CV: For performance evaluation
outer_cv = KFold(n_splits=5, shuffle=True, random_state=43)

# Parameter grid for Ridge alpha (regularization strength)
param_grid = {
    'model__alpha': np.logspace(-3, 3, 10) # Test values from 0.001 to 1000
}

# GridSearch 
clf = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=inner_cv,
    scoring=make_scorer(spearman_metric),
    n_jobs=-1
)

# Evaluation
nested_scores = cross_val_score(
    clf, 
    F_pairs, 
    human_sim, 
    cv=outer_cv, 
    scoring=make_scorer(spearman_metric),
    #n_jobs=-1
)

print(f"Nested CV Mean Spearman Rho: {np.mean(nested_scores):.3f} (±{np.std(nested_scores):.3f})")


### low-rank projection model

In [ ]:
class PairedStandardScaler(BaseEstimator, TransformerMixin):
    """Custom scaler MinMaxScaler"""
    def __init__(self):
        self.scaler = MinMaxScaler()
        
    def fit(self, X: np.ndarray, y=None):
        # Flatten (N_pairs, 2, N_features) -> (N_pairs * 2, N_features)
        self.scaler.fit(X.reshape(-1, X.shape[-1]))
        return self
        
    def transform(self, X: np.ndarray) -> np.ndarray:
        scaled_flat = self.scaler.transform(X.reshape(-1, X.shape[-1]))
        return scaled_flat.reshape(X.shape)

class low_rank_proj(BaseEstimator, RegressorMixin):
    def __init__(self, bottleneck_dim: int = 5, lambda_: float = 0.1, 
                 max_iter: int = 1000, lr: float = 0.001, clip_norm: float = 5.0):
        self.bottleneck_dim = bottleneck_dim
        self.lambda_ = lambda_
        self.max_iter = max_iter
        self.lr = lr
        self.clip_norm = clip_norm
        self.W_B = None
    
    def fit(self, F_pairs: np.ndarray, s: np.ndarray):
        n_pairs, n_features = F_pairs.shape[0], F_pairs.shape[2]
        rng = np.random.RandomState(42)
        
        # Initialize projection matrix
        self.W_B = rng.randn(self.bottleneck_dim, n_features) * 0.01
        
        prev_loss = np.inf
        
        for epoch in range(self.max_iter):
            L_total = 0.0
            grad = np.zeros_like(self.W_B)
            
            # Vectorized forward pass
            for idx in range(n_pairs):
                F_i = F_pairs[idx, 0]
                F_j = F_pairs[idx, 1]
                s_ij = s[idx]
                
                # Projections
                B_Fi = self.W_B @ F_i
                B_Fj = self.W_B @ F_j
                s_hat_ij = B_Fi.T @ B_Fj
                
                # Loss accumulation
                error = s_hat_ij - s_ij
                L_total += error ** 2
                
                # Gradient accumulation
                grad += 2 * error * (np.outer(B_Fj, F_i) + np.outer(B_Fi, F_j))
            
            # Regularization and Loss Averaging
            L_total = L_total / 2 + self.lambda_ * np.sum(self.W_B ** 2)
            grad = grad / n_pairs + 2 * self.lambda_ * self.W_B
            
            # Convergence check
            if abs(prev_loss - L_total) < 1e-6:
                break
            prev_loss = L_total
            
            # Gradient Clipping
            grad_norm = np.linalg.norm(grad)
            if np.isfinite(grad_norm) and grad_norm > self.clip_norm:
                grad *= self.clip_norm / (grad_norm + 1e-12)
            
            self.W_B -= self.lr * grad
            
        return self
    
    def predict(self, F_pairs: np.ndarray) -> np.ndarray:
        if self.W_B is None:
            raise RuntimeError("Model must be fit before prediction.")
            
        s_hat = []
        for idx in range(F_pairs.shape[0]):
            F_i = F_pairs[idx, 0]
            F_j = F_pairs[idx, 1]
            B_Fi = self.W_B @ F_i
            B_Fj = self.W_B @ F_j
            s_hat.append(B_Fi.T @ B_Fj)
        return np.array(s_hat)
    
def spearman_metric(y_true, y_pred):
    return spearmanr(y_true, y_pred).correlation

spearman_score = make_scorer(spearman_metric, greater_is_better=True)

In [ ]:
# CV pipeline
pipe = Pipeline([
    ('scaler', PairedStandardScaler()),
    ('simdr', low_rank_proj())
])

# 1. Define the dimensions to analyze
dims_to_test = [5] #[1,2,3,4,5,6,7,8,9,10,20,50,100]

print(f"{'Dim':<6} | {'Mean Spearman':<15} | {'Std Dev':<10}")
print("-" * 38)

for d in dims_to_test:
    
    # grid for INNER optimization
    param_grid_inner = {
        'simdr__lambda_': [0.1], # [0.001, 0.01, 0.1, 1.0, 10, 100],
        'simdr__lr': [0.001], # [0.001, 0.01, 0.1],
        'simdr__bottleneck_dim': [d]  # Fixed for this run
    }

    # Inner CV: Finds best lambda/lr for this specific d
    # This prevents training data leakage into hyperparameter selection
    inner_cv = KFold(n_splits=6, shuffle=True, random_state=42)
    
    clf = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid_inner,
        cv=inner_cv,
        scoring=spearman_score,
        n_jobs=-1,
        verbose=0
    )
    
    # Outer CV: Estimates generalization error
    # Evaluates the "process of tuning a d-dimensional model" on unseen data
    outer_cv = KFold(n_splits=6, shuffle=True, random_state=43)
    
    nested_scores = cross_val_score(
        clf, 
        F_pairs, 
        human_sim, 
        cv=outer_cv, 
        scoring=spearman_score,
        n_jobs=-1
    )
    
    # Record metrics
    mean_score = np.mean(nested_scores)
    std_score = np.std(nested_scores)
    
    print(f"{d:<6} | {mean_score:.4f}          | {std_score:.4f}")

In [ ]:
# Pipeline with Best Hyperparameters
final_pipe = Pipeline([
    ('scaler', PairedStandardScaler()),
    ('simdr', low_rank_proj(
        bottleneck_dim=5, 
        lambda_=0.1, 
        lr=0.001
    ))
])

# Train on all available data
print("Training final model on full dataset...")
final_pipe.fit(F_pairs, human_sim)

# save model
with open(output_path + 'projection_model.pkl', 'wb') as f:
    pickle.dump(final_pipe, f)

## predictions of similarities on the whole dataset

In [ ]:
# predicted_sim is the matrix of predicted similarities for all pairs of terms, 
# computed using the final model.
size = len(aligned_frequent)
predicted_sim = np.zeros((size, size))
for i in range(size):
    for j in range(i+1, size):
        f_i = aligned_frequent.iloc[i]['embeddings']
        f_j = aligned_frequent.iloc[j]['embeddings']
        pair = np.array([[f_i, f_j]])
        predicted_sim[i, j] = final_pipe.predict(pair)[0]
        predicted_sim[j, i] = predicted_sim[i, j]  # Symmetric matrix

In [ ]:
# normalize each row with a softmax
predicted_sim_normalized = np.zeros_like(predicted_sim)
for i in range(size):
    predicted_sim_normalized[i] = softmax(predicted_sim[i])

# does each row sum to 1? boolean check
row_sums = predicted_sim_normalized.sum(axis=1)
print("All rows sum to 1:", np.allclose(row_sums, 1.0))

# display heatmap of predicted similarities
plt.figure()
sns.heatmap(predicted_sim_normalized, cmap='viridis', cbar=True)
plt.show()

In [ ]:
# save predicted similarities
np.save(output_path + 'predicted_similarities.npy', predicted_sim_normalized)

# 2) Building alignment tables

In [ ]:
# to alignement tables 
pairs = [('fr', 'en'), ('fr', 'de'), ('fr', 'sr')]
tables = []
dataframes = []
data = aligned_frequent.reset_index(drop=True)

for source, target in pairs:
    lexicon = data[f'{target}_norm'].unique()
    source_terms = data[f'{source}_norm']
    ids = data['hash']
    source_hash = [f"{term}_{ID}" for ID,term in zip(ids, source_terms)]
    
    translations = np.zeros((len(data), len(lexicon)), dtype=int)
    print(translations.shape)

    for i,row in data.iterrows():
        target_term = row[f'{target}_norm']
        if target_term in lexicon:
            j = lexicon.tolist().index(target_term)
            translations[i,j] += 1
        else :
            print(f"Skipping target term: {target_term}")
            pass

    tables.append(translations)
    dataframes.append(pd.DataFrame(translations, index=source_hash, columns=lexicon))

# assert each row sums to 1
for df in dataframes:
    assert all(df.sum(axis=1) == 1)

dataframes[0]

In [ ]:
# save tables
with open(output_path + 'translation_tables.pkl', 'wb') as f:
    pickle.dump(tables, f)

# 3) LLM translations processing

In [ ]:
df_gptmini = pd.read_pickle(resources_path + 'gptmini_translations.pkl')

# keep only rows where en_norm freq > 5
en_norm_freq = df_gptmini['en_norm'].value_counts()
frequent_en_norm = en_norm_freq[en_norm_freq > 5].index
df_gptmini = df_gptmini[df_gptmini['en_norm'].isin(frequent_en_norm)]

de_norm_freq = df_gptmini['de_norm'].value_counts()
frequent_de_norm = de_norm_freq[de_norm_freq > 5].index
df_gptmini = df_gptmini[df_gptmini['de_norm'].isin(frequent_de_norm)]

sr_norm_freq = df_gptmini['sr_norm'].value_counts()
frequent_sr_norm = sr_norm_freq[sr_norm_freq > 5].index
df_gptmini = df_gptmini[df_gptmini['sr_norm'].isin(frequent_sr_norm)]

# remove rows where any of the norms is None or empty
df_gptmini = df_gptmini[
    df_gptmini['en_norm'].notna() & (df_gptmini['en_norm'] != '') & (df_gptmini['en_norm'] != 'None') &
    df_gptmini['de_norm'].notna() & (df_gptmini['de_norm'] != '') & (df_gptmini['de_norm'] != 'None') &
    df_gptmini['sr_norm'].notna() & (df_gptmini['sr_norm'] != '') & (df_gptmini['sr_norm'] != 'None')
]
print(f"Size after filtering: {len(df_gptmini)}")

# merge with aligned_frequent on hash
df_gptmini = pd.merge(aligned_frequent[[ 'hash', 'fr_norm', 'embeddings']], df_gptmini, on='hash', how='inner')


In [ ]:
# predicting similarities for gptmini data
size = len(df_gptmini)
predicted_sim_gptmini = np.zeros((size, size))
for i in range(size):
    for j in range(i+1, size):
        f_i = df_gptmini.iloc[i]['embeddings']
        f_j = df_gptmini.iloc[j]['embeddings']
        pair = np.array([[f_i, f_j]])
        predicted_sim_gptmini[i, j] = final_pipe.predict(pair)[0]
        predicted_sim_gptmini[j, i] = predicted_sim_gptmini[i, j]

predicted_sim_gptmini_normalized = np.zeros_like(predicted_sim_gptmini)
for i in range(size):
    predicted_sim_gptmini_normalized[i] = softmax(predicted_sim_gptmini[i])

# does each row sum to 1? boolean check
row_sums_gptmini = predicted_sim_gptmini_normalized.sum(axis=1)
print("All rows sum to 1 (GPT-Mini):", np.allclose(row_sums_gptmini, 1.0))

In [ ]:
# alignement tables from df_gptmini
pairs = [('fr', 'en'), ('fr', 'de'), ('fr', 'sr')]
gpt_tables = []
gpt_dataframes = []
for source, target in pairs:
    lexicon = df_gptmini[f'{target}_norm'].unique()
    source_terms = df_gptmini[f'{source}_norm']
    ids = df_gptmini.index
    source_hash = [f"{term}_{ID}" for ID,term in zip(ids, source_terms)]
    
    translations = np.zeros((len(df_gptmini), len(lexicon)), dtype=int)
    print(translations.shape)
    for i,row in df_gptmini.iterrows():
        target_term = row[f'{target}_norm']
        if target_term in lexicon:
            j = lexicon.tolist().index(target_term)
            translations[i,j] += 1
        else :
            print(f"Skipping target term: {target_term}")
            pass
    gpt_tables.append(translations)
    gpt_dataframes.append(pd.DataFrame(translations, columns=lexicon))

# assert each row sums to 1
for df in gpt_dataframes:
    assert all(df.sum(axis=1) == 1)

In [ ]:
# saving outputs
with open(output_path + 'predicted_similarities_gptmini.npy', 'wb') as f:
    np.save(f, predicted_sim_gptmini_normalized)
with open(output_path + 'translation_tables_gptmini.pkl', 'wb') as f:
    pickle.dump(gpt_tables, f)

# 4) IB analysis

In [ ]:
from tools import *

In [ ]:
sim = np.load(output_path + 'predicted_similarities.npy')
with open(output_path + 'translation_tables.pkl', 'rb') as f:
    tables = pickle.load(f)

sim_gpt = np.load(output_path + 'predicted_similarities_gptmini.npy')
with open(output_path + 'translation_tables_gptmini.pkl', 'rb') as f:
    gpt_tables = pickle.load(f)

## locating attested and generated translations in the information plane

In [ ]:
M = sim.shape[0]
pM = np.asarray([1/M]*M, dtype=float)
INFOS = []
COMPS = []

for encoder in tables:
    COMPS.append(complexity(pM, encoder))
    INFOS.append(accuracy( pM, encoder, sim))

In [ ]:
# LLM translations
M_gpt = sim_gpt.shape[0]
pM_gpt = np.asarray([1/M_gpt]*M_gpt, dtype=float)
INFOS_gpt = []
COMPS_gpt = []
for encoder in gpt_tables:
    COMPS_gpt.append(complexity(pM_gpt, encoder))
    INFOS_gpt.append(accuracy(pM_gpt, encoder, sim_gpt))

In [ ]:
plt.figure(figsize=(4, 2))
plt.scatter(COMPS, INFOS, color='blue')
plt.scatter(COMPS_gpt, INFOS_gpt, color='orange')
for comp, info, pair in zip(COMPS, INFOS, pairs):
    plt.text(comp, info, f'{pair[0]}->{pair[1]}', fontsize=6, ha='left')
for comp, info, pair in zip(COMPS_gpt, INFOS_gpt, pairs):
    plt.text(comp, info, f'{pair[0]}->{pair[1]} (GPT-Mini)', fontsize=6, ha='left')
plt.xlabel('Complexity I(M;W)')
plt.ylabel('Informativity I(W;U)')
plt.title('Information Plane of Language Pairs')
plt.grid(False)
plt.show()

## counter-factual translations

`W_RANGE` is the range of vocabulary size

`N_RANDOM` is the number of random encoders @ vocabulary size = W_RANGE

default : 40 * 100

In [ ]:
W_RANGE = 40
N_RANDOM = 100

def tailored_random_encoder(M, W) -> np.ndarray:
    """
    generate a random matrix of shape (M, W) = `translations.shape`
    by randomly generating M one hot encodings of size W,
    where M is the number of rows and W the number of columns of `translations`
    """ 
    rand_matrix = np.zeros((M, W), dtype=int)
    for i in range(M):
        j = np.random.randint(W)
        rand_matrix[i, j] = 1
    return rand_matrix

In [ ]:
# Generate random Translations and compute their complexity and informativity
RANDOM_INFOS = []
RANDOM_COMPS = []

for w in range(1,W_RANGE+1):
    for _ in range(N_RANDOM):
        random_encoder = tailored_random_encoder(M, w)
        RANDOM_COMPS.append(complexity(pM, random_encoder))
        RANDOM_INFOS.append(accuracy(pM, random_encoder, sim))

# mean accuracy and complexity of random Translations
mean_random_info = np.mean(RANDOM_INFOS)
mean_random_comp = np.mean(RANDOM_COMPS)
print(f"Mean Random Informativity: {mean_random_info:.6f}, Mean Random Complexity: {mean_random_comp:.6f}")

# std
std_random_info = np.std(RANDOM_INFOS)
std_random_comp = np.std(RANDOM_COMPS)
print(f"Std Random Informativity: {std_random_info:.6f}, Std Random Complexity: {std_random_comp:.6f}")

plt.figure(figsize=(2, 2))
plt.scatter(RANDOM_COMPS, RANDOM_INFOS, color='lightgray', alpha=0.5, label='Random Translations')
plt.scatter(COMPS, INFOS, color='blue', label='Attested Translations')
plt.xlabel('Complexity I(M;W)')
plt.ylabel('Informativity I(W;U)')
plt.title('Information Plane with Random Translations')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(False)
plt.show()
    

`N_PERMUTATIONS` is the number of random **permutated** counter-factual translations @ `ratio`

In [ ]:
N_PERMUTATIONS = 100

def random_rows_permutations(translations, ratio):
    """
    generate a random matrix of shape (M, W) = `translations.shape`
    by permuting a fraction `ratio` of rows in the original `translation` matrix
    """
    M, W = translations.shape
    n_changes = int(M * ratio)
    row_idxs = np.random.choice(M, n_changes, replace=False)
    permuted_rows = np.random.permutation(row_idxs)
    
    rand_matrix = translations.copy()
    rand_matrix[row_idxs] = translations[permuted_rows]
    
    return rand_matrix

In [ ]:
# Generate partially permuted Translations and compute their complexity and informativity
one_p_permutations= []
five_p_permutations = []
ten_p_permutations = []

for table in tables:
    one_p_permutations.append([])
    five_p_permutations.append([])
    ten_p_permutations.append([])
    for _ in range(N_PERMUTATIONS):
        one_p_permutations[-1].append(random_rows_permutations(table, 0.01))
        five_p_permutations[-1].append(random_rows_permutations(table, 0.05))
        ten_p_permutations[-1].append(random_rows_permutations(table, 0.1))

# compute accuracy of each permutation level
one_p_accuracies = []
five_p_accuracies = []
ten_p_accuracies = []

for table, one_p_list, five_p_list, ten_p_list in zip(tables, one_p_permutations, five_p_permutations, ten_p_permutations):
    one_p_accuracies.append([accuracy(pM, encoder, sim) for encoder in one_p_list])
    five_p_accuracies.append([accuracy(pM, encoder, sim) for encoder in five_p_list])
    ten_p_accuracies.append([accuracy(pM, encoder, sim) for encoder in ten_p_list])

In [ ]:
plt.figure(figsize=(2, 2))
plt.scatter(RANDOM_COMPS, RANDOM_INFOS, color='lightgray', alpha=0.5, label='Random Translations')
plt.scatter(COMPS, INFOS, color='blue', label='Attested Translations')
plt.scatter([comp for comp in COMPS for _ in range(N_PERMUTATIONS)],
            [acc for accs in one_p_accuracies for acc in accs],
            color='orange',
            alpha=0.1,
            s=10,
            label='1% Permutations')
plt.scatter([comp for comp in COMPS for _ in range(N_PERMUTATIONS)],
            [acc for accs in five_p_accuracies for acc in accs],
            color='red',
            alpha=0.1,
            s=10,
            label='5% Permutations')
plt.scatter([comp for comp in COMPS for _ in range(N_PERMUTATIONS)],
            [acc for accs in ten_p_accuracies for acc in accs],
            color='darkred',
            alpha=0.1,
            s=10,
            label='10% Permutations')
plt.xlabel('Complexity I(M;W)')
plt.ylabel('Informativity I(W;U)')
plt.title('Information Plane with Random Translations and Permutations')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(False)
plt.show()

## frontier calculation

In [ ]:
PRECISION = 1e-16
from tools import *

`BETAS` log-linearly spaced between 1 and $2^{20}$

In [ ]:
BETAS = np.logspace(20,1,num=100,base=2)

In [ ]:
Q_BETAS = [] # list of optimal encoders for each beta
opti_comp, opti_info = [], []

q_init = tailored_random_encoder(M, 40)


for beta in tqdm(BETAS, desc="Calculating optimal points", total=len(BETAS)):
    q_beta = BA_iterations(pM, sim, q_init, beta,)
    opti_comp.append(complexity(pM, q_beta))
    opti_info.append(accuracy(pM, q_beta, sim))
    Q_BETAS.append(q_beta)
    q_init = q_beta
    print(f"beta: {beta}, informativity: {opti_info[-1]}, complexity: {opti_comp[-1]}")


In [ ]:
curve = pd.DataFrame(
    data = {
        'beta': BETAS,
        'informativity' : opti_info,
        'complexity' : opti_comp,
        'J' : np.array(opti_comp) - np.array(BETAS)*np.array(opti_info),
        'q': Q_BETAS
        }
)

# save curve
with open(output_path + 'ib_curve_100.pkl', 'wb') as f:
    pickle.dump(curve, f)

In [ ]:
plt.figure(figsize=(2,2))
plt.plot(curve["complexity"], curve["informativity"], color='red', label='Optimal IB Frontier', marker='x', markersize=3, linestyle='')
plt.scatter(COMPS, INFOS, color='blue', label='Language Pairs')
plt.scatter(COMPS_gpt, INFOS_gpt, color='orange', label='GPT Translations')
plt.xlabel('Complexity')
plt.ylabel('Informativity')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

## $\epsilon$ distance to optimal encoder

In [ ]:
def find_epsilon(pM, pU_M, pW_M, curve_complexity, curve_informativity, BETAS):
    Fb = complexity(pM, pW_M) - BETAS * accuracy(pM, pW_M, pU_M)
    Fb_star= curve_complexity - BETAS * curve_informativity
    diff = Fb - Fb_star
    return(diff.min()/BETAS[diff.argmin()])

In [ ]:
attested_epsilon = []

for encoder,pair in zip(tables, pairs):
    epsilon = find_epsilon(
        pM,
        sim,
        encoder,
        curve["complexity"],
        curve["informativity"],
        BETAS
    )
    print(f"Efficiency loss (epsilon) for pair {pair}: {epsilon:.6f}")
    attested_epsilon.append(epsilon)

In [ ]:
# Compute epsilon for all pair and all perturbation levels
EN_DEVIATIONS_ONE = []
for fake_encoder in one_p_permutations[0] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    EN_DEVIATIONS_ONE.append(epsilon)

EN_DEVIATIONS_FIVE = []
for fake_encoder in five_p_permutations[0] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    EN_DEVIATIONS_FIVE.append(epsilon)

EN_DEVIATIONS_TEN = []
for fake_encoder in ten_p_permutations[0] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    EN_DEVIATIONS_TEN.append(epsilon)

DE_DEVIATIONS_ONE = []
for fake_encoder in one_p_permutations[1] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    DE_DEVIATIONS_ONE.append(epsilon)
DE_DEVIATIONS_FIVE = []
for fake_encoder in five_p_permutations[1] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    DE_DEVIATIONS_FIVE.append(epsilon)
DE_DEVIATIONS_TEN = []
for fake_encoder in ten_p_permutations[1] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    DE_DEVIATIONS_TEN.append(epsilon)

SR_DEVIATIONS_ONE = []
for fake_encoder in one_p_permutations[2] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    SR_DEVIATIONS_ONE.append(epsilon)
SR_DEVIATIONS_FIVE = []
for fake_encoder in five_p_permutations[2] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    SR_DEVIATIONS_FIVE.append(epsilon)
SR_DEVIATIONS_TEN = []
for fake_encoder in ten_p_permutations[2] :
    epsilon = np.abs(find_epsilon(pM, sim, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    SR_DEVIATIONS_TEN.append(epsilon)

In [ ]:

RANDOM_TRANSLATIONS_DEVIATIONS = []
for random_comp, random_info in zip(RANDOM_COMPS, RANDOM_INFOS):
    Fb = random_comp - BETAS * random_info
    Fb_star= curve['complexity'] - BETAS * curve['informativity']
    diff = Fb - Fb_star
    eps = diff.min()/BETAS[diff.argmin()]
    RANDOM_TRANSLATIONS_DEVIATIONS.append(eps)

mean_random_deviation = np.mean(RANDOM_TRANSLATIONS_DEVIATIONS)
std_random_deviation = np.std(RANDOM_TRANSLATIONS_DEVIATIONS)

In [ ]:
gpt_mini_deviations = []
for encoder in gpt_tables:
    epsilon = find_epsilon(pM_gpt, sim_gpt, encoder, curve["complexity"], curve["informativity"], BETAS)
    gpt_mini_deviations.append(epsilon)

In [ ]:
# dict for plotting
means = {
    'Attested': attested_epsilon,
    '1%': [np.mean(EN_DEVIATIONS_ONE), np.mean(DE_DEVIATIONS_ONE), np.mean(SR_DEVIATIONS_ONE)],
    '5%': [np.mean(EN_DEVIATIONS_FIVE), np.mean(DE_DEVIATIONS_FIVE), np.mean(SR_DEVIATIONS_FIVE)],
    '10%': [np.mean(EN_DEVIATIONS_TEN), np.mean(DE_DEVIATIONS_TEN), np.mean(SR_DEVIATIONS_TEN)],
}
stds = {
    'Attested': [0.0]*3,
    '1%': [np.std(EN_DEVIATIONS_ONE), np.std(DE_DEVIATIONS_ONE), np.std(SR_DEVIATIONS_ONE)],
    '5%': [np.std(EN_DEVIATIONS_FIVE), np.std(DE_DEVIATIONS_FIVE), np.std(SR_DEVIATIONS_FIVE)],
    '10%': [np.std(EN_DEVIATIONS_TEN), np.std(DE_DEVIATIONS_TEN), np.std(SR_DEVIATIONS_TEN)],
}

x = np.arange(len(pairs))
width = 0.15
original_col = '#009E73'
colors = ['#D55E00', '#0072B2', '#E69F00']


fig, ax = plt.subplots()

# attested translations
ax.bar(x - width, means['Attested'], width, label='Attested translations', color=original_col)

# permutated translations
ax.bar(x, means['1%'], width, yerr=stds['1%'], capsize=2, label='1% Perturbation', color=colors[0])
ax.bar(x + width, means['5%'], width, yerr=stds['5%'], capsize=2, label='5% Perturbation', color=colors[1])
ax.bar(x + 2*width, means['10%'], width, yerr=stds['10%'], capsize=2, label='10% Perturbation', color=colors[2])

# GPT-Mini translations
ax.bar(x + width*3, gpt_mini_deviations, width,
       label='GPT-Mini translations', color='#CC79A7')

#random translations
extra_x = len(pairs) + 0.5

ax.bar(
    extra_x,
    mean_random_deviation,
    width,
    yerr=std_random_deviation,
    capsize=2,
    label='Random translations',
    color='grey'
)

ax.set_xticks(list(x + 1.5*width) + [extra_x], list(pairs) + ['Random'])
ax.set_xlim(-0.5, extra_x + 0.5)


ax.set_ylabel('Deviation from optimality $\epsilon_q$', fontsize=7)
ax.tick_params(axis='y', labelsize=7)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=7)
plt.tight_layout()
plt.show()